In [0]:
from pyspark.sql.functions import (
    col, count, when, datediff, current_date,
    max as spark_max, sum as spark_sum,
    avg, lit
)
from pyspark.sql.functions import least, greatest
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("imports done")

In [0]:
orders_df   = spark.table("bharatmart.silver.slv_orders")
sessions_df = spark.table("bharatmart.silver.slv_sessions")
cart_df   = spark.table("bharatmart.silver.slv_cart_events")
reviews_df  = spark.table("bharatmart.silver.slv_reviews")
refunds_df  = spark.table("bharatmart.silver.slv_refunds")
payments_df = spark.table("bharatmart.silver.slv_payments")

print("tables loaded")

**Drop Ghost Orders**

In [0]:
clean_orders = orders_df.filter(col("customer_id").isNotNull())

print(f"clean orders : {clean_orders.count():,}")

**Recompute Total Amount**

In [0]:
# total_amount was 99% NULL in silver — recompute from order_amount
clean_orders = clean_orders.withColumn(
    "total_amount", col("order_amount") - col("discount_amount")
)

print("total_amount recomputed")

**RFM Base Features**

In [0]:
rfm = clean_orders.groupBy("customer_id").agg(
    spark_max("order_date").alias("last_order_date"),
    count("order_id").alias("total_orders"),
    spark_sum("total_amount").alias("total_spent"),
    avg("total_amount").alias("avg_order_value")
)

rfm = rfm.withColumn(
    "recency_days", datediff(current_date(), col("last_order_date"))
)

print(f"rfm customers : {rfm.count():,}")

**Clip and Round RFM Features**

In [0]:
p99_orders = rfm.approxQuantile("total_orders", [0.99], 0.01)[0]
p99_spent  = rfm.approxQuantile("total_spent",  [0.99], 0.01)[0]

rfm = rfm.withColumn("total_orders", least(col("total_orders"), lit(p99_orders)))
rfm = rfm.withColumn("total_spent", least(col("total_spent"),  lit(p99_spent)))
rfm = rfm.withColumn("total_spent", col("total_spent").cast("decimal(10,2)"))
rfm = rfm.withColumn("avg_order_value", col("avg_order_value").cast("decimal(10,2)"))

print("rfm features clipped and rounded")

**Refund Features**

In [0]:
refund_features = refunds_df.groupBy("customer_id").agg(
    count("refund_id").alias("refund_count")
)

rfm = rfm.join(refund_features, "customer_id", "left")
rfm = rfm.fillna(0, subset=["refund_count"])
rfm = rfm.withColumn(
    "refund_rate", (col("refund_count") / col("total_orders")).cast("decimal(5,2)")
)

print("refund features done")

**Cart Abandonment Feature**

In [0]:
cart_adds = cart_df.filter(col("action") == "cart_add") \
    .groupBy("customer_id") \
    .agg(count("event_id").alias("cart_adds"))

cart_feat = cart_adds.join(
    rfm.select("customer_id", "total_orders"), "customer_id", "left"
)

cart_feat = cart_feat.withColumn(
    "abandon_rate",
    greatest(lit(0.0), least(lit(1.0),
        (col("cart_adds") - col("total_orders")) / col("cart_adds")
    ))
).select("customer_id", "abandon_rate")

rfm = rfm.join(cart_feat, "customer_id", "left")
rfm = rfm.fillna(0.0, subset=["abandon_rate"])

print("cart abandonment done")

**Session Features**

In [0]:
session_feat = sessions_df.groupBy("customer_id").agg(
    avg("pages_viewed").alias("avg_pages_viewed"),
    avg("session_duration_seconds").alias("avg_session_duration"),
    count("session_id").alias("session_count")
)

rfm = rfm.join(session_feat, "customer_id", "left")
rfm = rfm.fillna(0, subset=["avg_pages_viewed", "avg_session_duration", "session_count"])
rfm = rfm.withColumn("avg_pages_viewed", col("avg_pages_viewed").cast("decimal(5,2)"))
rfm = rfm.withColumn("avg_session_duration", col("avg_session_duration").cast("decimal(8,2)"))

print("session features done")

**Channel and Payment Features**

In [0]:
channel_feat = sessions_df.groupBy("customer_id").agg(
    avg(when(col("channel") == "organic_search", 1).otherwise(0)).alias("pct_organic"),
    avg(when(col("channel") == "social_media",   1).otherwise(0)).alias("pct_social")
)

order_payments = clean_orders.select("order_id", "customer_id") \
    .join(payments_df.select("order_id", "payment_method"), "order_id", "left")

payment_counts = order_payments.filter(col("payment_method").isNotNull()) \
    .groupBy("customer_id", "payment_method") \
    .agg(count("order_id").alias("cnt"))

window = Window.partitionBy("customer_id").orderBy(col("cnt").desc())

payment_feat = payment_counts \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .select("customer_id", col("payment_method").alias("preferred_payment"))

rfm = rfm.join(channel_feat,  "customer_id", "left")
rfm = rfm.join(payment_feat,  "customer_id", "left")
rfm = rfm.fillna(0.0,   subset=["pct_organic", "pct_social"])
rfm = rfm.fillna("UPI", subset=["preferred_payment"])

print("channel and payment features done")

**Review Rating Feature**

In [0]:
rating_feat = reviews_df.groupBy("customer_id").agg(
    avg("rating").alias("avg_rating")
)

rfm = rfm.join(rating_feat, "customer_id", "left")
rfm = rfm.fillna(3.0, subset=["avg_rating"])
rfm = rfm.withColumn("avg_rating", col("avg_rating").cast("decimal(3,1)"))

print("review rating done")

**Add Churn Label**

In [0]:
rfm = rfm.withColumn(
    "churn_label",
    when((col("recency_days") > 90) & (col("total_orders") >= 2), 1).otherwise(0)
)

churn_count  = rfm.filter(col("churn_label") == 1).count()
active_count = rfm.filter(col("churn_label") == 0).count()

print(f"churned : {churn_count:,}")
print(f"active  : {active_count:,}")

**Encode Payment and Select Final Features**

In [0]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="preferred_payment",
    outputCol="payment_idx"
)

rfm = indexer.fit(rfm).transform(rfm)

feature_cols = [
    "customer_id",
    "recency_days", "total_orders", "total_spent", "avg_order_value",
    "refund_count", "refund_rate", "abandon_rate",
    "avg_pages_viewed", "avg_session_duration", "session_count",
    "pct_organic", "pct_social", "avg_rating", "payment_idx",
    "churn_label"
]

final_df = rfm.select(feature_cols)

print(f"final rows : {final_df.count():,}")
print(f"final columns  : {len(feature_cols)}")

**Overwrite Feature Table**

In [0]:
final_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bharatmart.ml.churn_features")

count = spark.table("bharatmart.ml.churn_features").count()
print(f"wrote {count:,} rows to bharatmart.ml.churn_features")